# MarianMT — Medical Fine-Tuning on MeSpEn
**iDISC Personalized Translation Engines · Phase 2: Domain Adaptation**

Fine-tunes `Helsinki-NLP/opus-mt-es-en` on the **MeSpEn** parallel corpus (Barcelona Supercomputing Center)  
using the **MedlinePlus health-topics** subset — 726 KB download, ~2 K sentence pairs, trains in minutes.

---
| | Base model | Fine-tuned model |
|---|---|---|
| Model | `Helsinki-NLP/opus-mt-es-en` | same + MeSpEn adapter weights |
| Domain | General | Medical (MedlinePlus) |
| Training data | OPUS (general) | MeSpEn subset (~500 pairs) |

> **MeSpEn reference:** Villegas et al. (2018). *MeSpEn: A Medical Spanish-English parallel corpus*. Zenodo. https://doi.org/10.5281/zenodo.3562536

In [1]:
%%capture
!pip install transformers datasets sentencepiece sacrebleu accelerate

## 1 — Download MeSpEn (MedlinePlus subset)

In [2]:
import os, tarfile, urllib.request

DATA_DIR   = "mespen_data"
ARCHIVE    = "MedlinePlus-health_topics-dublin_core-Sp-En.tar.bz2"
ZENODO_URL = f"https://zenodo.org/records/3562536/files/{ARCHIVE}"

os.makedirs(DATA_DIR, exist_ok=True)
local_path = os.path.join(DATA_DIR, ARCHIVE)

if not os.path.exists(local_path):
    print(f"Downloading MeSpEn MedlinePlus (~726 KB)…")
    urllib.request.urlretrieve(ZENODO_URL, local_path)
    print("Download complete.")
else:
    print("Archive already present — skipping download.")

# Extract
extract_dir = os.path.join(DATA_DIR, "medlineplus_extracted")
if not os.path.exists(extract_dir):
    print("Extracting…")
    with tarfile.open(local_path, "r:bz2") as tf:
        tf.extractall(extract_dir, filter="data")
    print("Extraction complete.")
else:
    print("Already extracted.")

# List top-level contents
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = "  " * (level + 1)
        for f in files[:5]:
            print(f"{subindent}{f}")
        if len(files) > 5:
            print(f"{subindent}... ({len(files)} files total)")
    break  # only show top level in first pass

Download complete.
Extracting…
Extraction complete.
medlineplus_extracted/


## 2 — Parse Dublin-Core XML → Parallel Pairs

In [3]:
import xml.etree.ElementTree as ET
import glob

DC_NS  = "http://purl.org/dc/elements/1.1/"
XML_NS = "http://www.w3.org/XML/1998/namespace"

def extract_pairs_from_xml(xml_path):
    """Return list of (es, en) pairs from a MeSpEn Dublin-Core XML file."""
    pairs = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return pairs

    for field in ["title", "description"]:
        texts = {}
        tag = f"{{{DC_NS}}}{field}"
        for elem in root.iter(tag):
            if elem.get("type") == "alternate":   # skip alternate titles
                continue
            lang = elem.get(f"{{{XML_NS}}}lang", "")[:2].lower()
            text = (elem.text or "").strip()
            if lang in ("es", "en") and text:
                texts[lang] = text
        if "es" in texts and "en" in texts:
            pairs.append((texts["es"], texts["en"]))

    return pairs

# Walk all XML files in the extracted archive
xml_files = glob.glob(os.path.join(extract_dir, "**", "*.xml"), recursive=True)
print(f"Found {len(xml_files)} XML files")

all_pairs = []
for path in xml_files:
    all_pairs.extend(extract_pairs_from_xml(path))

print(f"Extracted {len(all_pairs)} ES<->EN parallel pairs")
print()
print("Sample pairs:")
for es, en in all_pairs[:3]:
    print(f"  ES: {es[:90]}")
    print(f"  EN: {en[:90]}")
    print()


Found 1062 XML files
Extracted 2030 ES<->EN parallel pairs

Sample pairs:
  ES: Ataxia de Friedreich
  EN: Friedreich's Ataxia

  ES: La ataxia de Freidreich es una enfermedad hereditaria que daña el sistema nervioso. Afecta
  EN: Friedreich's ataxia is an inherited disease that damages your nervous system. The damage a

  ES: Enfermedades peritoneales
  EN: Peritoneal Disorders



## 3 — Dataset Exploration

In [4]:
import pandas as pd

df = pd.DataFrame(all_pairs, columns=["es", "en"])
df["es_len"] = df["es"].str.split().str.len()
df["en_len"] = df["en"].str.split().str.len()

print(f"Total pairs   : {len(df)}")
print(f"ES word length: mean={df.es_len.mean():.1f}  median={df.es_len.median():.0f}  max={df.es_len.max()}")
print(f"EN word length: mean={df.en_len.mean():.1f}  median={df.en_len.median():.0f}  max={df.en_len.max()}")
print()

print("── Sample pairs ──────────────────────────────────────────────────────")
for _, row in df.head(5).iterrows():
    print(f"ES: {row.es[:120]}")
    print(f"EN: {row.en[:120]}")
    print()

Total pairs   : 2030
ES word length: mean=96.2  median=9  max=1201
EN word length: mean=85.9  median=6  max=1090

── Sample pairs ──────────────────────────────────────────────────────
ES: Ataxia de Friedreich
EN: Friedreich's Ataxia

ES: La ataxia de Freidreich es una enfermedad hereditaria que daña el sistema nervioso. Afecta la médula espinal y los nervi
EN: Friedreich's ataxia is an inherited disease that damages your nervous system. The damage affects your spinal cord and th

ES: Enfermedades peritoneales
EN: Peritoneal Disorders

ES: El peritoneo es el tejido que recubre la pared abdominal y cubre la mayor parte de los órganos en el abdomen. Un líquido
EN: Your peritoneum is the tissue that lines your abdominal wall and covers most of the organs in your abdomen. A liquid, pe

ES: Histoplasmosis
EN: Histoplasmosis



## 4 — Data Preparation

Filter by length, deduplicate, then take a **small training subset** (fast run).

In [5]:
# ── Config ────────────────────────────────────────────────────────────────────
MIN_WORDS  = 3
MAX_WORDS  = 200   # descriptions avg ~190 words; tokenizer truncates to 128 tokens
N_TRAIN    = 500   # small subset — change to 1500+ for better quality
N_EVAL     = 100
RANDOM_SEED = 42
# ─────────────────────────────────────────────────────────────────────────────

# Filter
df_clean = (
    df
    .query("es_len >= @MIN_WORDS and es_len <= @MAX_WORDS")
    .query("en_len >= @MIN_WORDS and en_len <= @MAX_WORDS")
    .drop_duplicates(subset=["es"])
    .reset_index(drop=True)
)

print(f"After filtering: {len(df_clean)} pairs (from {len(df)})")

# Shuffle + split (percentage-safe)
df_shuffled = df_clean.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

n_avail  = len(df_shuffled)
n_train  = min(N_TRAIN, int(n_avail * 0.8))
n_eval   = min(N_EVAL, n_avail - n_train)

df_train = df_shuffled.iloc[:n_train]
df_eval  = df_shuffled.iloc[n_train : n_train + n_eval]

print(f"Train : {len(df_train)} pairs")
print(f"Eval  : {len(df_eval)} pairs")
print()
print("Sample training pair:")
row = df_train.iloc[0]
print(f"  ES: {row.es[:120]}")
print(f"  EN: {row.en[:120]}")


After filtering: 981 pairs (from 2030)
Train : 500 pairs
Eval  : 100 pairs

Sample training pair:
  ES: Cuando se siente mareado, puede sentirse aturdido, confundido o desorientado. Si siente que la habitación está girando, 
  EN: When you're dizzy, you may feel lightheaded, woozy, or disoriented. If you feel like you or the room are spinning, you h


## 5 — Tokenize for MarianMT

In [6]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset

MODEL_NAME = "Helsinki-NLP/opus-mt-es-en"
MAX_LEN    = 128

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    model_inputs = tokenizer(
        batch["es"], max_length=MAX_LEN, truncation=True, padding=False
    )
    labels = tokenizer(
        text_target=batch["en"], max_length=MAX_LEN, truncation=True, padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_dict({"es": df_train["es"].tolist(), "en": df_train["en"].tolist()})
eval_ds  = Dataset.from_dict({"es": df_eval["es"].tolist(),  "en": df_eval["en"].tolist()})

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["es", "en"])
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=["es", "en"])

print(f"Tokenized train: {len(train_tok)} examples")
print(f"Tokenized eval : {len(eval_tok)} examples")
print(f"Sample input_ids length: {len(train_tok[0]['input_ids'])} tokens")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenized train: 500 examples
Tokenized eval : 100 examples
Sample input_ids length: 128 tokens


## 6 — Baseline Quality (Pre-Fine-Tuning)

Record translations from the base model on eval sentences — used as comparison after training.

In [7]:
import sacrebleu, time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

base_model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

def batch_translate(model, texts, batch_size=16):
    all_translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, num_beams=4)
        all_translations.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_translations

# Use first 50 eval pairs as held-out test set
test_es  = df_eval["es"].tolist()[:50]
test_ref = df_eval["en"].tolist()[:50]

t0 = time.perf_counter()
base_translations = batch_translate(base_model, test_es)
base_time = (time.perf_counter() - t0) * 1000

base_bleu = sacrebleu.corpus_bleu(base_translations, [test_ref]).score

print(f"Base model BLEU (medical eval): {base_bleu:.2f}")
print(f"Translation time: {base_time:.0f} ms for {len(test_es)} sentences\n")

print("── Sample base translations ──────────────────────────────")
for es, ref, hyp in zip(test_es[:3], test_ref[:3], base_translations[:3]):
    print(f"ES : {es[:100]}")
    print(f"REF: {ref[:100]}")
    print(f"HYP: {hyp[:100]}")
    print()

Device: cuda



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Base model BLEU (medical eval): 31.73
Translation time: 7429 ms for 50 sentences

── Sample base translations ──────────────────────────────
ES : La enfermedad de Charcot-Marie-Tooth (CMT, por sus siglas en inglés) es un grupo de trastornos nervi
REF: Charcot-Marie-Tooth disease (CMT) is a group of genetic nerve disorders. It is named after the three
HYP: Charcot-Marie-Tooth’s disease (CMT) is a group of genetic nervous disorders. Its name is due to the 

ES : Salud sexual del adolescente
REF: Teen Sexual Health
HYP: Adolescent sexual health

ES : Torceduras y distensiones
REF: Sprains and Strains
HYP: Torces and strains



## 7 — Fine-Tuning on MeSpEn

In [8]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

ft_model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)

OUTPUT_DIR = "./mespen_medical_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=True,
    fp16=(device == "cuda"),
    logging_steps=20,
    load_best_model_at_end=True,
    report_to="none",
    label_smoothing_factor=0.1,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=ft_model, padding=True)

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

print(f"Training on {len(train_tok)} pairs, {training_args.num_train_epochs} epochs")
steps_per_epoch = len(train_tok) // training_args.per_device_train_batch_size
print(f"~{steps_per_epoch} steps/epoch → ~{steps_per_epoch * training_args.num_train_epochs} total steps")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Training on 500 pairs, 3 epochs
~62 steps/epoch → ~186 total steps


In [9]:
train_result = trainer.train()

print(f"\nTraining complete.")
print(f"  Runtime : {train_result.metrics['train_runtime']:.1f} s")
print(f"  Loss    : {train_result.metrics['train_loss']:.4f}")

Epoch,Training Loss,Validation Loss
1,5.462497,5.163874
2,5.169034,5.122106
3,4.980753,5.122043


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



Training complete.
  Runtime : 48.4 s
  Loss    : 5.1750


## 8 — Save Fine-Tuned Model

In [10]:
ft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./mespen_medical_model


## 9 — Evaluation: Base vs Fine-Tuned

In [11]:
ft_model.eval()

ft_translations = batch_translate(ft_model, test_es)
ft_bleu = sacrebleu.corpus_bleu(ft_translations, [test_ref]).score

print("═" * 55)
print(f"  Base model BLEU  : {base_bleu:.2f}")
print(f"  Fine-tuned BLEU  : {ft_bleu:.2f}")
delta = ft_bleu - base_bleu
sign  = "+" if delta >= 0 else ""
print(f"  Delta            : {sign}{delta:.2f}")
print("═" * 55)

═══════════════════════════════════════════════════════
  Base model BLEU  : 31.73
  Fine-tuned BLEU  : 33.63
  Delta            : +1.90
═══════════════════════════════════════════════════════


In [12]:
# Qualitative side-by-side comparison
GREEN = "\033[92m"
RESET = "\033[0m"

# Pick a few interesting medical examples
sample_indices = list(range(min(5, len(test_es))))

for i in sample_indices:
    print(f"{'─'*65}")
    print(f"ES   : {test_es[i]}")
    print(f"REF  : {test_ref[i]}")
    print(f"BASE : {base_translations[i]}")
    print(f"FT   : {GREEN}{ft_translations[i]}{RESET}")
    b = sacrebleu.sentence_bleu(base_translations[i], [test_ref[i]]).score
    f = sacrebleu.sentence_bleu(ft_translations[i],   [test_ref[i]]).score
    print(f"       BLEU base={b:.1f}  ft={f:.1f}")
    print()

─────────────────────────────────────────────────────────────────
ES   : La enfermedad de Charcot-Marie-Tooth (CMT, por sus siglas en inglés) es un grupo de trastornos nerviosos genéticos. Su nombre se debe a los tres médicos que la identificaron por primera vez. En los Estados Unidos, la CMT afecta aproximadamente a 1 de cada 2.500 personas. La enfermedad CMT afecta los nervios periféricos. Los nervios periféricos conducen las señales de movimiento y de sensaciones entre el cerebro, la médula espinal y el resto del cuerpo. Los síntomas suelen comenzar durante la adolescencia. Los problemas en los pies, tales como el pie arqueado (arco pronunciado) o los dedos en martillo, pueden ser los primeros síntomas. A medida que la CMT avanza, la parte inferior de las piernas puede debilitarse. Más adelante, las manos también pueden debilitarse. Los médicos diagnostican esta enfermedad mediante un examen neurológico, pruebas genéticas o biopsias de los nervios. No existe una cura. La enfermedad 

## 10 — Quick Medical Terminology Test

Qualitative spot-check on domain-specific sentences not in the training set.

In [13]:
medical_probes = [
    "El paciente presenta síntomas de hipertensión arterial y diabetes mellitus tipo 2.",
    "Se recomienda realizar una colonoscopia cada cinco años a partir de los cincuenta años.",
    "La resonancia magnética mostró una lesión en el lóbulo frontal del cerebro.",
    "El tratamiento con quimioterapia se suspendió debido a efectos adversos graves.",
    "Los niveles de glucosa en sangre en ayunas estaban por encima de los valores normales.",
]

base_probes = batch_translate(base_model, medical_probes)
ft_probes   = batch_translate(ft_model,   medical_probes)

for i, es in enumerate(medical_probes):
    print(f"{'─'*65}")
    print(f"ES  : {es}")
    print(f"BASE: {base_probes[i]}")
    print(f"FT  : {GREEN}{ft_probes[i]}{RESET}")
    print()

─────────────────────────────────────────────────────────────────
ES  : El paciente presenta síntomas de hipertensión arterial y diabetes mellitus tipo 2.
BASE: The patient has symptoms of hypertension and type 2 diabetes mellitus.
FT  : The patient has symptoms of hypertension and type 2 diabetes mellitus.

─────────────────────────────────────────────────────────────────
ES  : Se recomienda realizar una colonoscopia cada cinco años a partir de los cincuenta años.
BASE: A colonoscopy is recommended every five years from the age of fifty.
FT  : A colonoscopy is recommended every five years from age 50.

─────────────────────────────────────────────────────────────────
ES  : La resonancia magnética mostró una lesión en el lóbulo frontal del cerebro.
BASE: MRI showed an injury to the frontal lobe of the brain.
FT  : MRI showed an injury to the frontal lobe of the brain.

─────────────────────────────────────────────────────────────────
ES  : El tratamiento con quimioterapia se suspendió 

---
## Summary

| Step | Detail |
|---|---|
| Dataset | MeSpEn — MedlinePlus health topics (Zenodo 3562536) |
| Training pairs | 500 (from ~2 K available) |
| Base model | `Helsinki-NLP/opus-mt-es-en` |
| Fine-tuned model | saved to `./mespen_medical_model` |
| Next step | Increase N_TRAIN to 1500+ and add PubMed subset for better coverage |

> **To use more MeSpEn data:** replace the 726 KB MedlinePlus file with  
> `Pubmed-dublin_core-Sp-En.tar.bz2` (84 MB, 127 K pairs) from the same Zenodo record.